# Reproduce the paper's figures and tables

Regenerates every float in **Uncertainty Quantification for Aquatic Ecotoxicity
Prediction** from the saved model artefacts, in manuscript order.

### Prerequisites

1. `pip install -r requirements.txt`
2. Place the ADORE files in `data/raw/` (see `data/raw/README.md`)
3. Cross-validated run: `python scripts/train_bfm.py --n_folds 5 --n_iter 2000 --n_burn 100`
4. Pair-grouped run, for Table 2 only:
   `python scripts/train_bfm.py --n_folds 5 --group_key pair --out_dir outputs/models_pairCV`
5. Full-data run: `python scripts/generate_predictions.py --n_iter 2000 --n_burn 100`

Supporting Information floats are produced outside this notebook; see the last section.

## Setup

In [ ]:
import sys
from pathlib import Path

# Project root (one level up from notebooks/)
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR / "src"))
sys.path.insert(0, str(ROOT_DIR / "analysis"))

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

## Configuration

In [ ]:
# Exposure duration the figures are drawn at
DURATION_HOURS = 48

# SSD case studies (Figures 9 and 10)
SSD_CHEMICALS = {
    "Atrazine": "1912-24-9",
    "Chlorfenprop-methyl": "14437-17-3",
}

# Hazardous-concentration percentile (Figures 11 and 12)
HC_PERCENTILE = 20

# Posterior curves drawn per ensemble SSD
N_SSD_CURVES = 2000

## Load data and model artefacts

In [ ]:
import numpy as np
import pandas as pd
from data.load_ecotox import load_ecotox_data

DATA_DIR = ROOT_DIR / "data" / "raw"
MODELS_DIR = ROOT_DIR / "outputs" / "models"

# Load raw dataset
full_data, y_centered, y_mean = load_ecotox_data(
    adore_path=DATA_DIR / "ecotox_mortality_processed.csv",
    chemicals_path=DATA_DIR / "ecotox_properties_with-oecd-function.csv",
    use_molar=False,
    use_selfies=False, use_mol2vec=False, use_fingerprint=False,
    shuffle=True, random_state=42,
)
full_data["y_true"] = y_centered + y_mean

print(f"Loaded {len(full_data):,} observations")
print(f"  Unique chemicals: {full_data['CAS'].nunique():,}")
print(f"  Unique species:   {full_data['species'].nunique():,}")
print(f"  Centering mean:   {y_mean:.4f} log mg/L")

In [ ]:
# Load out-of-fold predictions (from cross-validation)
oof_mean = np.load(MODELS_DIR / "oof_mean.npy")
oof_epistemic = np.load(MODELS_DIR / "oof_epistemic.npy")
oof_aleatoric = np.load(MODELS_DIR / "oof_aleatoric.npy")

df = full_data.copy()
df["y_pred"] = oof_mean + y_mean
df["epistemic_var"] = oof_epistemic
df["aleatoric_var"] = oof_aleatoric
df["total_var"] = df["epistemic_var"] + df["aleatoric_var"]
df["epistemic_sd"] = np.sqrt(df["epistemic_var"])
df["aleatoric_sd"] = np.sqrt(df["aleatoric_var"])
df["total_sd"] = np.sqrt(df["total_var"])

print("Loaded OOF predictions.")

In [ ]:
# Load full prediction matrix (from full-dataset training)
pred_df = pd.read_parquet(MODELS_DIR / "full_predictions.parquet")
pred_df["epistemic_sd"] = np.sqrt(pred_df["pred_epistemic_var"])
pred_df["aleatoric_sd"] = np.sqrt(pred_df["pred_aleatoric_var"])

print(f"Loaded {len(pred_df):,} full predictions.")

---
## Dataset characterization

### Table 1: summary statistics

In [ ]:
from dataset_figures import dataset_summary_table

dataset_summary_table(full_data)

### Figure 1: rank-frequency of observations

In [ ]:
from dataset_figures import plot_rank_frequency

plot_rank_frequency(full_data)

### Figure 2: within-triplet replicate SDs

In [ ]:
from dataset_figures import plot_replicate_sd_distribution

plot_replicate_sd_distribution(full_data)

---
## Predictive accuracy

### Table 2: RMSE under both cross-validation grouping schemes

The triplet-grouped run is the default `outputs/models/`. The pair-grouped run is
the like-for-like comparison with Posthuma et al. and must be produced separately
with `--group_key pair` (step 4 above); the cell skips it if absent.

In [ ]:
rmse_triplet = np.sqrt(np.mean((oof_mean - y_centered) ** 2))
print(f"Chemical-species-duration triplet, this work : {rmse_triplet:.3f} log mg/L")

pair_dir = ROOT_DIR / "outputs" / "models_pairCV"
if (pair_dir / "oof_mean.npy").exists():
    oof_pair = np.load(pair_dir / "oof_mean.npy")
    rmse_pair = np.sqrt(np.mean((oof_pair - y_centered) ** 2))
    print(f"Chemical-species pair, this work             : {rmse_pair:.3f} log mg/L")
else:
    print("Chemical-species pair, this work             : not run (see step 4)")
print("Chemical-species pair, Posthuma et al.       : 0.85 log mg/L (published)")

### Figure 3: predicted vs measured

In [ ]:
from analyze_results import plot_predicted_vs_measured_correlation

plot_predicted_vs_measured_correlation(df, duration_hours=DURATION_HOURS)

### Figure 4: residual diagnostics

In [ ]:
from analyze_results import analyze_prediction_bias

analyze_prediction_bias(df, duration_hours=DURATION_HOURS)

---
## Uncertainty validation

### Table 3: aleatoric SD against empirical replicate SD

In [ ]:
from dataset_figures import replicate_calibration_table

replicate_calibration_table(full_data)

### Figure 5: uncertainty vs data availability

In [ ]:
from uncertainty_figures import plot_uncertainty_vs_observations

plot_uncertainty_vs_observations(df)

### Figure 6: composition of predictive variance

Also writes `variance_share_by_nobs.csv` and prints the global aleatoric share.

In [ ]:
import variance_decomposition

variance_decomposition.main()

### Figure 7: posterior predictive calibration

Needs the per-sample arrays `oof_pred_samples.npy` and `oof_aleatoric_samples.npy`.

In [ ]:
import calibration_figures

calibration_figures.main()

### Figure 8: example predictions with uncertainty

In [ ]:
from uncertainty_figures import plot_example_predictions

plot_example_predictions(df)

---
## Species Sensitivity Distributions

### Figures 9 and 10: per-species uncertainty, and the posterior ensemble

Figure 9 gives the uncertainty on each species prediction; Figure 10 gives the
uncertainty on the curve as a whole, with the traditional lognormal fit overlaid.

In [ ]:
from ssd_analysis import (
    set_target, filter_observations, filter_predictions,
    plot_novel_ssd_with_uncertainty,
)
from ssd_mc_uncertainty import plot_ssd_with_uncertainty

for chem_name, cas in SSD_CHEMICALS.items():
    print("\n" + "=" * 70)
    print(f"  {chem_name} (CAS {cas})")
    print("=" * 70)

    set_target(cas, chem_name)
    obs_c = filter_observations(full_data)
    pred_c = filter_predictions(pred_df)

    plot_novel_ssd_with_uncertainty(pred_c)                      # Figure 9
    plot_ssd_with_uncertainty(pred_c, obs_c, n_curves=N_SSD_CURVES)  # Figure 10

---
## Hazardous concentrations

### HC20 for all chemicals

Iterates every chemical over every posterior sample, so it is slow. The result is
cached to CSV and reused by the two figures below.

In [ ]:
hcx_csv = ROOT_DIR / "outputs" / "figures" / f"hcx_comparison_{DURATION_HOURS}h.csv"

if hcx_csv.exists():
    print(f"Loading cached HC{HC_PERCENTILE} comparison from {hcx_csv}")
    df_hcx = pd.read_csv(hcx_csv)
else:
    from ssd_mc_uncertainty import compute_hcx_all_chemicals, compute_traditional_hcx

    df_mc = compute_hcx_all_chemicals(percentiles=[HC_PERCENTILE])
    df_trad = compute_traditional_hcx(full_data, percentiles=[HC_PERCENTILE])
    df_hcx = df_mc.merge(df_trad, on="CAS", how="left")
    df_hcx.to_csv(hcx_csv, index=False)
    print(f"Saved {hcx_csv}")

print(f"{len(df_hcx):,} chemicals")

### Figure 11: HC20 forest plot

In [ ]:
from hcx_plots import plot_hc_forest

plot_hc_forest(df_hcx)

### Figure 12: traditional vs BFM HC20

In [ ]:
from hcx_plots import plot_hc_correlation

plot_hc_correlation(df_hcx)

---
## Supporting Information

Two SI floats and the SI's numerical claims are produced outside this notebook,
because they need either a separate sweep of model fits or the full posterior
sample arrays:

| SI item | Produced by |
|---|---|
| Rank-sensitivity table and figure | `python scripts/rank_sweep.py` (refits the model at k = 4, 8, 16, 32, 64) |
| Aleatoric vs pooled replicate variance; highest-count bin; b0 mixing | `python analysis/si_diagnostics/aleatoric_vs_replicates.py` |
| HC20 species-set decomposition; prediction-trace ESS | `python analysis/si_diagnostics/ssd_species_selection.py` |
| Null calibration ratio for Table 3 | `python analysis/si_diagnostics/null_calibration.py` |
